# Nomadic LLM Experiment — NB03: Nomadic Full Evaluation

## 이 노트북이 측정하는 것

| 모델 | 설명 |
|------|------|
| Nomadic_Full | 전체 시스템: LoRA + Δx + PolicyNet + DwellTime |

NB02의 baseline들과 같은 시나리오/프롬프트로 측정하여  
NB04에서 통합 비교 가능하게 저장한다.

**추가로 측정:**
- Parameter-matched 확인: Nomadic_Full vs Vanilla 파라미터 수 비교
- Switch saturation 모니터링 (PolicyNet stay/switch 분포 실시간 체크)

In [ ]:
# ============================================================
# STEP 0: 환경 로드
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, math
DRIVE_BASE = '/content/drive/MyDrive/nomadic_llm_results'
with open(os.path.join(DRIVE_BASE, 'latest_run_dir.txt')) as f:
    lines = f.read().strip().split('\n')
    RUN_DIR, MODEL_KEY = lines[0], lines[1]

with open(os.path.join(RUN_DIR, 'run_config.json')) as f:
    cfg = json.load(f)
MODEL_PATH = cfg['MODEL_PATH']
HIDDEN_DIM = cfg['HIDDEN_DIM']

with open(os.path.join(RUN_DIR, 'prompts.json'), encoding='utf-8') as f:
    P = json.load(f)
EVAL_PROMPTS          = P['EVAL_PROMPTS']
SCENARIO_B_SEQUENCE   = P['SCENARIO_B_SEQUENCE']
SCENARIO_B_SHIFT_POINT= P['SCENARIO_B_SHIFT_POINT']
SCENARIO_C_CLEAN      = P['SCENARIO_C_CLEAN']
SCENARIO_C_NOISY      = P['SCENARIO_C_NOISY']
SCENARIO_D_HYBRID     = P['SCENARIO_D_HYBRID']
SCENARIO_E_CASES      = P['SCENARIO_E_CASES']
SCENARIO_F_DENSE      = P['SCENARIO_F_DENSE']
SCENARIO_F_SPARSE     = P['SCENARIO_F_SPARSE']
SCENARIO_G_PROMPTS    = P['SCENARIO_G_PROMPTS']

with open(os.path.join(RUN_DIR, 'expert_paths.json')) as f:
    expert_paths = json.load(f)

print(f'RUN_DIR  : {RUN_DIR}')
print(f'Experts  : {list(expert_paths.keys())}')

In [ ]:
# ============================================================
# STEP 1: 패키지 + 모델 로드
# ============================================================
!pip install -q transformers==4.44.0 accelerate bitsandbytes peft

import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True)
base_model.eval()
print('✅ 모델 로드 완료')

In [ ]:
# ============================================================
# STEP 2: 컴포넌트 정의 + PolicyNet 로드
# ============================================================

class HybridDeltaTracker:
    def __init__(self, alpha=0.8, beta=0.85, tau_min=2.0, tau_max=8.0,
                 tau_var_scale=6.0, tau_window=8):
        self.alpha=alpha; self.beta=beta
        self.tau_min=tau_min; self.tau_max=tau_max
        self.tau_var_scale=tau_var_scale; self.tau_window=tau_window
        self.reset()
    def reset(self):
        self.prev_x=None; self.ema_err=0.0; self.baseline_err=0.0
        self.recent_de=deque(maxlen=self.tau_window); self.history=[]
    def compute(self, current_x, current_err):
        if self.prev_x is None: delta_env=0.0
        else:
            cos=F.cosine_similarity(current_x.float().flatten(),
                                    self.prev_x.float().flatten(), dim=0)
            delta_env=float((1.0-cos).clamp(0).item())
        self.prev_x=current_x.detach().clone()
        self.recent_de.append(delta_env)
        self.ema_err=self.alpha*self.ema_err+(1-self.alpha)*current_err
        self.baseline_err=self.beta*self.baseline_err+(1-self.beta)*current_err
        delta_err=max(0.0, self.ema_err-self.baseline_err)
        delta_hybrid=float(torch.tanh(torch.tensor(delta_env+delta_err)).item())
        sigma2=float(np.var(self.recent_de)) if len(self.recent_de)>=2 else 0.0
        tau_dyn=self.tau_min+(self.tau_max-self.tau_min)/(1.0+self.tau_var_scale*sigma2)
        rec=dict(delta_env=delta_env, delta_err=delta_err,
                 delta_hybrid=delta_hybrid, sigma2=sigma2, tau_dyn=tau_dyn)
        self.history.append(rec)
        return rec


class NomadicPolicyNet(nn.Module):
    NUM_EXPERTS=3; EXPERT_KEYS=['math','code','language']
    def __init__(self, hidden_dim, policy_hidden=128):
        super().__init__()
        self.proj=nn.Linear(hidden_dim,policy_hidden)
        self.shared=nn.Sequential(
            nn.Linear(policy_hidden+5,policy_hidden),nn.ReLU(),
            nn.Linear(policy_hidden,policy_hidden),nn.ReLU())
        self.stay_switch_head=nn.Linear(policy_hidden,2)
        self.target_head=nn.Linear(policy_hidden,self.NUM_EXPERTS)
        self.mode_head=nn.Linear(policy_hidden,2)
    def forward(self,hidden,meta_signals):
        if hidden.dim()==3: hidden=hidden.squeeze(1)
        h=F.relu(self.proj(hidden.float()))
        inp=torch.cat([h,meta_signals.float()],dim=-1)
        h=self.shared(inp)
        return(F.softmax(self.stay_switch_head(h),dim=-1),
               F.softmax(self.target_head(h),dim=-1),
               F.softmax(self.mode_head(h),dim=-1))


def build_meta_signals(rec, device):
    return torch.tensor([[
        rec['delta_hybrid'],rec['delta_err'],rec['delta_hybrid'],
        math.tanh(rec['sigma2']*10.0),
        math.tanh((rec['tau_dyn']-5.0)/5.0),
    ]],dtype=torch.float32,device=device)

def topk_entropy(logits,k=50):
    probs=F.softmax(logits,dim=-1)
    topk=probs.topk(k).values; topk=topk/topk.sum()
    return float(-(topk*(topk+1e-10).log()).sum().item())

def repeat_rate(tokens):
    if len(tokens)<3: return 0.0
    ngrams=[tuple(tokens[i:i+3]) for i in range(len(tokens)-2)]
    return 1.0-len(set(ngrams))/len(ngrams)

EXPERT_KEYS=['math','code','language']
NUM_EXPERTS=3
MAX_NEW_TOKENS=40
T_STABLE=0.35; T_TRANSITION=0.90
N_RUNS=3
DOMAIN_PHASE={'math':'stable','code':'stable','language':'transition'}

policy_net=NomadicPolicyNet(HIDDEN_DIM)
policy_net.load_state_dict(
    torch.load(os.path.join(RUN_DIR,'policy_net.pt'),
               map_location=base_model.device))
policy_net=policy_net.to(base_model.device)
policy_net.eval()

def _load_expert(key):
    m=PeftModel.from_pretrained(base_model,expert_paths[key])
    m.eval(); return m
def _unload_expert(m): del m; torch.cuda.empty_cache()

def _next_tok(logits,temp):
    probs=F.softmax(logits/max(temp,1e-4),dim=-1)
    tok=torch.multinomial(probs,1)
    lp=F.log_softmax(logits/max(temp,1e-4),dim=-1).gather(1,tok).item()
    return tok,lp

def base_result(text,entropies,log_probs,expert_trace=None,dx_trace=None):
    ppl=float(np.exp(-np.mean(log_probs))) if log_probs else float('nan')
    switches=sum(1 for i in range(1,len(expert_trace or []))
                 if (expert_trace or [])[i]!=(expert_trace or [])[i-1])
    tokens=tokenizer.encode(text)
    return{
        'text':text,
        'mean_entropy':float(np.mean(entropies)) if entropies else float('nan'),
        'perplexity':ppl,
        'switch_rate':switches/max(1,len(expert_trace or [1])),
        'repeat_rate':repeat_rate(tokens),
        'mean_dx':float(np.mean(dx_trace)) if dx_trace else 0.0,
        'expert_trace':expert_trace or [],
        'dx_trace':dx_trace or [],
    }

print(f'✅ 컴포넌트 로드 완료 | PolicyNet params: {sum(p.numel() for p in policy_net.parameters()):,}')

In [ ]:
# ============================================================
# STEP 3: Nomadic_Full 생성 함수
# ============================================================

# Dwell-time 추적기
class DwellTracker:
    def __init__(self):
        self.current_expert = None
        self.dwell_count    = 0

    def update(self, expert_key):
        if expert_key == self.current_expert:
            self.dwell_count += 1
        else:
            self.current_expert = expert_key
            self.dwell_count    = 1
        return self.dwell_count


def generate_nomadic_full(prompt):
    """
    Nomadic Full:
    - HybridDeltaTracker → Δx 신호
    - PolicyNet → stay/switch + target expert + mode
    - DwellTracker → dwell-time 기반 switching pressure 조정
    - 온도: T_STABLE + (T_TRANSITION - T_STABLE) * delta_hybrid
    """
    tracker = HybridDeltaTracker()
    dwell   = DwellTracker()
    ids     = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    ents, lps, expert_trace, dxs = [], [], [], []
    switch_probs_log = []

    current_key   = 'math'
    current_model = _load_expert(current_key)

    for step in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = current_model(ids, output_hidden_states=True)
        logits = out.logits[:, -1, :]
        hidden = out.hidden_states[-1][:, -1, :]
        err    = float(1.0 - F.softmax(logits, dim=-1).max().item())
        rec    = tracker.compute(hidden, err)
        meta   = build_meta_signals(rec, base_model.device)

        with torch.no_grad():
            ss, tgt, mode = policy_net(hidden.unsqueeze(0), meta)

        switch_prob = ss[0, 1].item()
        tgt_key     = EXPERT_KEYS[tgt.argmax(-1).item()]
        hard_mode   = mode.argmax(-1).item() == 1
        d           = rec['delta_hybrid']
        tau_dyn     = rec['tau_dyn']

        # Dwell-time 기반 switching pressure
        dwell_count = dwell.update(current_key)
        if dwell_count > tau_dyn:
            # dwell 초과: switching pressure 증가
            effective_switch = switch_prob + 0.2
        else:
            effective_switch = switch_prob

        # Routing decision
        if effective_switch >= 0.5 or d >= 0.45:
            next_key = tgt_key
        else:
            next_key = current_key

        if next_key != current_key:
            _unload_expert(current_model)
            current_model = _load_expert(next_key)
            current_key   = next_key
            dwell.update(next_key)  # reset

        temp = T_STABLE + (T_TRANSITION - T_STABLE) * d
        # Hard routing: 안정 단계에서 greedy로 강제
        if hard_mode and d < 0.2:
            temp = 0.1

        ents.append(topk_entropy(logits / max(temp, 1e-4)))
        tok, lp = _next_tok(logits, temp)
        lps.append(lp)
        expert_trace.append(current_key)
        dxs.append(d)
        switch_probs_log.append(switch_prob)
        ids = torch.cat([ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id:
            break

    _unload_expert(current_model)
    text = tokenizer.decode(ids[0], skip_special_tokens=True)
    result = base_result(text, ents, lps, expert_trace=expert_trace, dx_trace=dxs)
    result['mean_switch_prob'] = float(np.mean(switch_probs_log))
    return result


print('✅ Nomadic_Full 생성 함수 정의 완료')

In [ ]:
# ============================================================
# STEP 4: Switch saturation 사전 체크
# ============================================================
# 측정 전 PolicyNet이 stable/transition을 구분하는지 확인.
# saturation 감지 시 결과 해석에 주의 필요.

print('PolicyNet switch saturation 체크...')
saturation_data = {}

for domain in ['math', 'language']:
    prompts = EVAL_PROMPTS[domain][:3]
    sw_probs = []
    expert = _load_expert(domain)
    tracker = HybridDeltaTracker()

    for prompt in prompts:
        ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
        tracker.reset()
        for _ in range(10):
            with torch.no_grad():
                out = expert(ids, output_hidden_states=True)
            logits = out.logits[:, -1, :]
            hidden = out.hidden_states[-1][:, -1, :]
            err    = float(1.0 - F.softmax(logits, dim=-1).max().item())
            rec    = tracker.compute(hidden, err)
            meta   = build_meta_signals(rec, base_model.device)
            with torch.no_grad():
                ss, _, _ = policy_net(hidden.unsqueeze(0), meta)
            sw_probs.append(ss[0, 1].item())
            tok = logits.argmax(-1, keepdim=True)
            ids = torch.cat([ids, tok], dim=-1)
            if tok.item() == tokenizer.eos_token_id: break

    _unload_expert(expert)
    mean_sp = float(np.mean(sw_probs))
    saturation_data[domain] = mean_sp
    flag = ' ⚠️ SATURATED' if mean_sp > 0.90 or mean_sp < 0.10 else ' ✅'
    print(f'  {domain:10s}: mean_switch_prob={mean_sp:.3f}{flag}')

sp_diff = abs(saturation_data['math'] - saturation_data['language'])
print(f'\n  |math - language| switch_prob diff: {sp_diff:.3f}')
if sp_diff < 0.1:
    print('  ⚠️  경고: 도메인 간 switch_prob 차이가 작음.')
    print('       아래 결과에서 ΔH 차이가 나타나더라도 PolicyNet 기여가 아닐 수 있음.')
    print('       NB04 분석에서 이 한계를 명시할 것.')

with open(os.path.join(RUN_DIR, 'saturation_check.json'), 'w') as f:
    json.dump({'saturation': saturation_data, 'sp_diff': sp_diff}, f, indent=2)

In [ ]:
# ============================================================
# STEP 5: 시나리오 A — 도메인별 기본 평가 (Nomadic_Full)
# ============================================================
print('시나리오 A: Nomadic_Full 도메인 평가...')
results_A = []

for domain in ['math', 'code', 'language']:
    prompts = EVAL_PROMPTS[domain]
    for prompt in prompts:
        run_ents, run_ppls, run_reps, run_sws, run_dxs, run_sp = [], [], [], [], [], []
        for run in range(N_RUNS):
            r = generate_nomadic_full(prompt)
            run_ents.append(r['mean_entropy'])
            run_ppls.append(r['perplexity'])
            run_reps.append(r['repeat_rate'])
            run_sws.append(r['switch_rate'])
            run_dxs.append(r['mean_dx'])
            run_sp.append(r['mean_switch_prob'])
        results_A.append({
            'model':        'Nomadic_Full',
            'domain':       domain,
            'phase':        DOMAIN_PHASE[domain],
            'prompt':       prompt[:50],
            'mean_entropy': float(np.mean(run_ents)),
            'std_entropy':  float(np.std(run_ents)),
            'perplexity':   float(np.mean(run_ppls)),
            'repeat_rate':  float(np.mean(run_reps)),
            'switch_rate':  float(np.mean(run_sws)),
            'mean_dx':      float(np.mean(run_dxs)),
            'mean_switch_prob': float(np.mean(run_sp)),
        })
    print(f'  {domain}: 완료')

df_A = pd.DataFrame(results_A)
df_A.to_csv(os.path.join(RUN_DIR, 'nomadic_scenario_A.csv'),
            index=False, encoding='utf-8-sig')
print(f'✅ 시나리오 A 완료')

In [ ]:
# ============================================================
# STEP 6: 시나리오 B, C, E, F, G (NB02와 동일 구조)
# ============================================================
# ── 시나리오 B ──────────────────────────────────────────────
print('시나리오 B: 도메인 급변 Δx...')
tracker_b = HybridDeltaTracker(); tracker_b.reset()
dx_trace_b = []
for prompt in SCENARIO_B_SEQUENCE:
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    with torch.no_grad():
        out = base_model(ids, output_hidden_states=True)
    hidden = out.hidden_states[-1][:, -1, :]
    err    = float(1.0 - F.softmax(out.logits[:, -1, :], dim=-1).max().item())
    rec    = tracker_b.compute(hidden, err)
    dx_trace_b.append(rec['delta_hybrid'])

shift_pt = SCENARIO_B_SHIFT_POINT
result_B = {
    'model': 'Nomadic_Full',
    'pre_mean_dx':  float(np.mean(dx_trace_b[:shift_pt])),
    'post_mean_dx': float(np.mean(dx_trace_b[shift_pt:])),
    'delta_dx':     float(np.mean(dx_trace_b[shift_pt:])) - float(np.mean(dx_trace_b[:shift_pt])),
}
print(f'  pre={result_B["pre_mean_dx"]:.4f} → post={result_B["post_mean_dx"]:.4f}  Δ={result_B["delta_dx"]:+.4f}')
pd.DataFrame([result_B]).to_csv(os.path.join(RUN_DIR, 'nomadic_scenario_B.csv'), index=False)

# ── 시나리오 C ──────────────────────────────────────────────
print('시나리오 C: 노이즈 면역력...')
results_C = []
for condition, prompts in [('clean', SCENARIO_C_CLEAN), ('noisy', SCENARIO_C_NOISY)]:
    ents, ppls = [], []
    for prompt in prompts:
        r = generate_nomadic_full(prompt)
        ents.append(r['mean_entropy']); ppls.append(r['perplexity'])
    results_C.append({'model':'Nomadic_Full','condition':condition,
                      'mean_entropy':float(np.mean(ents)),'perplexity':float(np.mean(ppls))})
pd.DataFrame(results_C).to_csv(os.path.join(RUN_DIR,'nomadic_scenario_C.csv'),index=False)

# ── 시나리오 D (Nomadic_Full만 해당) ────────────────────────
print('시나리오 D: 도메인 중첩...')
results_D = []
for prompt in SCENARIO_D_HYBRID:
    r = generate_nomadic_full(prompt)
    # oscillation: A→B→A 패턴 비율
    tr = r['expert_trace']
    osc = sum(1 for k in range(2,len(tr)) if tr[k]==tr[k-2] and tr[k]!=tr[k-1])
    osc_rate = osc / max(1, len(tr)-2)
    # dominant expert
    from collections import Counter
    dom_exp = Counter(tr).most_common(1)[0][0] if tr else 'n/a'
    results_D.append({'model':'Nomadic_Full','prompt':prompt[:50],
                      'oscillation':osc_rate,'dominant_expert':dom_exp,
                      'switch_rate':r['switch_rate'],'mean_dx':r['mean_dx']})
pd.DataFrame(results_D).to_csv(os.path.join(RUN_DIR,'nomadic_scenario_D.csv'),
                                index=False,encoding='utf-8-sig')

# ── 시나리오 E ──────────────────────────────────────────────
print('시나리오 E: 유혹-회복...')
results_E = []

def run_lure_scenario_full(cases):
    rows = []
    for math_prefix, lure, math_cont in cases:
        tracker = HybridDeltaTracker()
        # Baseline
        baseline_dxs = []
        ids = tokenizer(math_prefix,return_tensors='pt').input_ids.to(base_model.device)
        for _ in range(8):
            with torch.no_grad():
                out = base_model(ids,output_hidden_states=True)
            h = out.hidden_states[-1][:,-1,:]
            e = float(1.0-F.softmax(out.logits[:,-1,:],dim=-1).max().item())
            rec = tracker.compute(h,e)
            baseline_dxs.append(rec['delta_hybrid'])
            ids = torch.cat([ids,out.logits[:,-1,:].argmax(-1,keepdim=True)],dim=-1)
        baseline_mean = float(np.mean(baseline_dxs))
        # Lure spike
        lure_ids = tokenizer(math_prefix+lure,return_tensors='pt').input_ids.to(base_model.device)
        with torch.no_grad():
            out = base_model(lure_ids,output_hidden_states=True)
        h = out.hidden_states[-1][:,-1,:]
        e = float(1.0-F.softmax(out.logits[:,-1,:],dim=-1).max().item())
        lure_rec = tracker.compute(h,e)
        lure_spike = lure_rec['delta_hybrid'] - baseline_mean
        # Recovery
        rec_ids = tokenizer(math_prefix+lure+math_cont,
                            return_tensors='pt').input_ids.to(base_model.device)
        recovery_steps = 0
        for step in range(20):
            with torch.no_grad():
                out = base_model(rec_ids,output_hidden_states=True)
            h = out.hidden_states[-1][:,-1,:]
            e = float(1.0-F.softmax(out.logits[:,-1,:],dim=-1).max().item())
            rec = tracker.compute(h,e)
            recovery_steps+=1
            if rec['delta_hybrid'] <= baseline_mean+0.05: break
            tok = out.logits[:,-1,:].argmax(-1,keepdim=True)
            rec_ids = torch.cat([rec_ids,tok],dim=-1)
        rows.append({'model':'Nomadic_Full','baseline_dx':baseline_mean,
                     'lure_spike':lure_spike,'recovery_steps':recovery_steps})
    return rows

results_E = run_lure_scenario_full(SCENARIO_E_CASES)
avg_rec = float(np.mean([r['recovery_steps'] for r in results_E]))
print(f'  avg recovery_steps={avg_rec:.1f}')
pd.DataFrame(results_E).to_csv(os.path.join(RUN_DIR,'nomadic_scenario_E.csv'),
                                index=False,encoding='utf-8-sig')

# ── 시나리오 F ──────────────────────────────────────────────
print('시나리오 F: 정보 밀도...')
results_F = []
for density, prompts in [('dense',SCENARIO_F_DENSE),('sparse',SCENARIO_F_SPARSE)]:
    tracker=HybridDeltaTracker(); tracker.reset()
    dx_list,conf_list=[],[]
    for prompt in prompts:
        ids=tokenizer(prompt,return_tensors='pt').input_ids.to(base_model.device)
        with torch.no_grad():
            out=base_model(ids,output_hidden_states=True)
        h=out.hidden_states[-1][:,-1,:]
        e=float(1.0-F.softmax(out.logits[:,-1,:],dim=-1).max().item())
        rec=tracker.compute(h,e)
        dx_list.append(rec['delta_hybrid']); conf_list.append(e)
    corr=float(np.corrcoef(dx_list,conf_list)[0,1]) if len(dx_list)>1 else 0.0
    results_F.append({'model':'Nomadic_Full','density':density,
                      'mean_dx':float(np.mean(dx_list)),
                      'mean_conf':float(np.mean(conf_list)),'corr_dx_conf':corr})
pd.DataFrame(results_F).to_csv(os.path.join(RUN_DIR,'nomadic_scenario_F.csv'),index=False)

# ── 시나리오 G ──────────────────────────────────────────────
print('시나리오 G: 장기 고착도...')
results_G = []
LONG_STEPS=60; CONV_THRESHOLD=0.02
for domain, prompt in SCENARIO_G_PROMPTS.items():
    # Nomadic_Full로 long generation
    tracker=HybridDeltaTracker()
    expert=_load_expert(domain)  # 해당 도메인 expert 사용
    ids=tokenizer(prompt,return_tensors='pt').input_ids.to(base_model.device)
    phi_trace=[]; conv_step=LONG_STEPS
    for step in range(LONG_STEPS):
        with torch.no_grad():
            out=expert(ids,output_hidden_states=True)
        h=out.hidden_states[-1][:,-1,:]
        e=float(1.0-F.softmax(out.logits[:,-1,:],dim=-1).max().item())
        rec=tracker.compute(h,e)
        phi_trace.append(rec['delta_hybrid'])
        if rec['delta_hybrid']<=CONV_THRESHOLD and conv_step==LONG_STEPS:
            conv_step=step
        tok=out.logits[:,-1,:].argmax(-1,keepdim=True)
        ids=torch.cat([ids,tok],dim=-1)
        if tok.item()==tokenizer.eos_token_id: break
    _unload_expert(expert)
    phi_arr=np.array(phi_trace)
    slope=float(np.polyfit(np.arange(len(phi_arr)),phi_arr,1)[0]) if len(phi_arr)>5 else 0.0
    results_G.append({'model':'Nomadic_Full','domain':domain,
                      'convergence_step':conv_step,
                      'final_phi_mean':float(np.mean(phi_trace[-5:])) if len(phi_trace)>=5 else float('nan'),
                      'decay_rate':slope})
pd.DataFrame(results_G).to_csv(os.path.join(RUN_DIR,'nomadic_scenario_G.csv'),index=False)
print('✅ NB03 완료. 다음: NB04_analysis.ipynb')